# Ансамбль детекторов
**YOLOv8 + Faster-RCNN + RT-DETR** → **Weighted Box Fusion**

# Импорт библиотек

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm.auto import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO, RTDETR
from ensemble_boxes import weighted_boxes_fusion

from torchmetrics.detection import MeanAveragePrecision
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torchvision.ops import nms

# Константы

In [ ]:
CLASSES = {
    0: "Aortic enlargement", 1: "Atelectasis",      2: "Calcification",
    3: "Cardiomegaly",       4: "Consolidation",    5: "ILD",
    6: "Infiltration",       7: "Lung Opacity",     8: "Nodule/Mass",
    9: "Other lesion",      10: "Pleural effusion", 11: "Pleural thickening",
   12: "Pneumothorax",      13: "Pulmonary fibrosis",
}
CLASS_MAP = {name: cid for cid, name in CLASSES.items()}
CLASS_MAP["No finding"] = 14

# Конфигурация

In [ ]:
IMG_DIR = Path("data/processed/train")
TRAIN_CSV = Path("data/processed/train_balanced.csv")
VAL_CSV = Path("data/processed/val_split.csv")
WEIGHTS_DIR = Path("weights")

BATCH_SIZE  = 2
MAX_SAMPLES = None
IOU_THR     = 0.4

CFG = {
    # Пути к весам детекторов
    "weights_yolov8": WEIGHTS_DIR / "yolov8.pt",
    "weights_faster_rcnn": WEIGHTS_DIR / "rt_detr.pt",
    "weights_rt_detr": WEIGHTS_DIR / "faster_rcnn.pth",

    # Размеры входа детекторов
    "image_size_det":    1024,
    "image_size_rcnn":   1024,
    "image_size_rtdetr":  800,

    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Пороги детекторов (занижены для Recall)
    "yolov8_conf":      0.05,
    "faster_rcnn_conf": 0.05,
    "rt_detr_conf":     0.05,

    # WBF
    "wbf_iou_thr":     0.60,
    "wbf_skip_thr":    0.0001,
    "final_score_thr": 0.01,

    # Веса моделей в WBF
    "w_yolov8":      0.4286,
    "w_faster_rcnn": 0.1429,
    "w_rt_detr":     0.4286,
}
print(f"Устройство: {CFG['device']}")

# Обработка изображений

In [ ]:
def load_image_for_detectors(path: Path) -> Image.Image:
    arr = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if arr is None:
        raise FileNotFoundError(f"Не удалось открыть: {path}")
    if arr.ndim == 3:
        arr = arr[:, :, 0]
    arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(arr)

# Детекторы

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

class YOLOv8Detector:
    def __init__(self, cfg: dict):
        self.conf = cfg["yolov8_conf"]
        self.imgsz = cfg["image_size_det"]
        self.device = cfg["device"]
        self.model = YOLO(cfg["weights_yolov8"])

    def predict(self, images: List[Image.Image], ids: List[str]) -> List[Dict]:
        results = self.model.predict(
            source=images, conf=self.conf, iou=0.40,
            imgsz=self.imgsz, device=self.device, verbose=False,
        )
        return [
            {"image_id": iid, "img_w": img.size[0], "img_h": img.size[1],
             "boxes": r.boxes.xyxy.cpu().numpy(),
             "scores": r.boxes.conf.cpu().numpy(),
             "labels": r.boxes.cls.cpu().numpy().astype(int)}
            for img, iid, r in zip(images, ids, results)
        ]


class FasterRCNNDetector:
    def __init__(self, cfg: dict):
        from torchvision.models.detection import FasterRCNN
        from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
        from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

        self.conf   = cfg["faster_rcnn_conf"]
        self.device = torch.device(cfg["device"])

        backbone = resnet_fpn_backbone("resnet50", pretrained=False)
        model = FasterRCNN(backbone, num_classes=15)
        in_feat = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, 15)
        model.roi_heads.score_thresh = self.conf
        model.roi_heads.nms_thresh = 0.50
        model.roi_heads.detections_per_img = 300

        weights_path = Path(cfg["weights_faster_rcnn"])
        state = torch.load(weights_path, map_location=self.device)
        model.load_state_dict(state.get("model_state_dict", state))

        self.model = model.to(self.device).eval()

        self.transform = A.Compose([
            A.Resize(cfg["image_size_rcnn"], cfg["image_size_rcnn"]),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2()
        ])

    def predict(self, images: List[Image.Image], ids: List[str]) -> List[Dict]:
        tensors = []
        orig_sizes = []
        for img in images:
            arr = np.array(img.convert("RGB"))
            orig_sizes.append((img.size[0], img.size[1]))
            t = self.transform(image=arr)["image"]
            tensors.append(t.to(self.device))

        with torch.no_grad():
            raw = self.model(tensors)

        out = []
        for (orig_w, orig_h), iid, r in zip(orig_sizes, ids, raw):
            scores = r["scores"].cpu().numpy()
            boxes = r["boxes"].cpu().numpy()
            labels = r["labels"].cpu().numpy().astype(int) - 1

            size = self.transform.transforms[0].height
            boxes[:, [0, 2]] = boxes[:, [0, 2]] * orig_w / size
            boxes[:, [1, 3]] = boxes[:, [1, 3]] * orig_h / size

            mask = scores >= self.conf
            out.append({
                "image_id": iid,
                "img_w": orig_w,
                "img_h": orig_h,
                "boxes": boxes[mask],
                "scores": scores[mask],
                "labels": labels[mask],
            })
        return out


class RTDETRDetector:
    def __init__(self, cfg: dict):
        self.conf = cfg["rt_detr_conf"]
        self.imgsz = cfg["image_size_rtdetr"]
        self.device = cfg["device"]
        self.model = RTDETR(cfg["weights_rt_detr"])

    def predict(self, images: List[Image.Image], ids: List[str]) -> List[Dict]:
        results = self.model.predict(
            source=images, conf=self.conf,
            imgsz=self.imgsz, device=self.device, verbose=False,
        )
        return [
            {"image_id": iid, "img_w": img.size[0], "img_h": img.size[1],
             "boxes": r.boxes.xyxy.cpu().numpy(),
             "scores": r.boxes.conf.cpu().numpy(),
             "labels": r.boxes.cls.cpu().numpy().astype(int)}
            for img, iid, r in zip(images, ids, results)
        ]


# WBF

In [ ]:
def run_wbf(
    preds_per_model: List[List[Dict]],
    weights: List[float],
    iou_thr: float,
    skip_thr: float,
    final_thr: float,
) -> List[Dict]:

    results = []
    for i in range(len(preds_per_model[0])):
        img_preds = [m[i] for m in preds_per_model]
        img_w, img_h = img_preds[0]["img_w"], img_preds[0]["img_h"]
        image_id = img_preds[0]["image_id"]

        boxes_list, scores_list, labels_list = [], [], []
        for p in img_preds:
            b = p["boxes"].copy().astype(np.float32)
            s = p["scores"].copy().astype(np.float32)
            l = p["labels"].copy().astype(np.float32)
            if len(b) == 0:
                boxes_list.append(np.zeros((0,4), dtype=np.float32))
                scores_list.append(np.zeros(0, dtype=np.float32))
                labels_list.append(np.zeros(0, dtype=np.float32))
                continue
            b[:, [0,2]] = np.clip(b[:, [0,2]] / img_w, 0, 1)
            b[:, [1,3]] = np.clip(b[:, [1,3]] / img_h, 0, 1)
            b = np.stack([np.minimum(b[:,0],b[:,2]), np.minimum(b[:,1],b[:,3]),
                          np.maximum(b[:,0],b[:,2]), np.maximum(b[:,1],b[:,3])], axis=1)
            valid = (b[:,2]-b[:,0]) * (b[:,3]-b[:,1]) > 1e-6
            boxes_list.append(b[valid]); scores_list.append(s[valid]); labels_list.append(l[valid])

        if all(len(b) == 0 for b in boxes_list):
            results.append({"image_id": image_id, "boxes": np.zeros((0,4)),
                            "scores": np.zeros(0), "labels": np.zeros(0, dtype=int)})
            continue

        fb, fs, fl = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights, iou_thr=iou_thr, skip_box_thr=skip_thr,
        )
        keep = fs >= final_thr
        fb, fs, fl = fb[keep], fs[keep], fl[keep].astype(int)
        fb[:, [0,2]] *= img_w;  fb[:, [1,3]] *= img_h
        results.append({"image_id": image_id, "boxes": fb.astype(np.float32),
                        "scores": fs, "labels": fl})
    return results

# Ансамбль

In [ ]:
class Ensemble:
    def __init__(self, cfg: dict):
        self.cfg = cfg
        self.yolo = YOLOv8Detector(cfg)
        self.faster_rcnn = FasterRCNNDetector(cfg)
        self.rt_detr = RTDETRDetector(cfg)

    def predict(
        self,
        images_det: List[Image.Image],
        image_ids: Optional[List[str]] = None,
    ) -> List[Dict]:
        if image_ids is None:
            image_ids = [str(i) for i in range(len(images_det))]

        fused = run_wbf(
            preds_per_model=[
                self.yolo.predict(images_det, image_ids),
                self.faster_rcnn.predict(images_det, image_ids),
                self.rt_detr.predict(images_det, image_ids),
            ],
            weights=[self.cfg["w_yolov8"], self.cfg["w_faster_rcnn"], self.cfg["w_rt_detr"]],
            iou_thr=self.cfg["wbf_iou_thr"],
            skip_thr=self.cfg["wbf_skip_thr"],
            final_thr=self.cfg["final_score_thr"],
        )

        results = []
        for entry in fused:
            iid = entry["image_id"]
            boxes = np.clip(entry["boxes"], 0, None)
            scores = entry["scores"]
            labels = entry["labels"]
            results.append({
                "image_id": iid,
                "boxes": boxes,
                "scores": scores,
                "labels": labels,
                "class_names": [CLASSES.get(int(l), f"cls_{l}") for l in labels],
                "has_pathology": len(boxes) > 0,
            })
        return results

    def update(self, **kwargs) -> None:
        self.cfg.update(kwargs)
        self.yolo.conf = self.cfg["yolov8_conf"]
        self.faster_rcnn.conf = self.cfg["faster_rcnn_conf"]
        self.faster_rcnn.model.roi_heads.score_thresh = self.cfg["faster_rcnn_conf"]
        self.rt_detr.conf = self.cfg["rt_detr_conf"]


# Метрики

In [ ]:
def compute_metrics(preds: List[Dict], targets: List[Dict], iou_thr: float = 0.4) -> Dict:
    metric = MeanAveragePrecision(
        iou_thresholds=[iou_thr],
        rec_thresholds=torch.linspace(0, 1, 101).tolist(),
        class_metrics=True,
        box_format="xyxy",
    )
    metric.update(
        [{"boxes": torch.tensor(p["boxes"], dtype=torch.float32),
          "scores": torch.tensor(p["scores"], dtype=torch.float32),
          "labels": torch.tensor(p["labels"], dtype=torch.int64)} for p in preds],
        [{"boxes": torch.tensor(t["boxes"], dtype=torch.float32),
          "labels": torch.tensor(t["labels"], dtype=torch.int64)} for t in targets],
    )
    r = metric.compute()
    map_val = float(r["map"])
    map_pc = r.get("map_per_class", torch.zeros(14))

    total_tp = total_fp = total_fn = 0
    recall_per_class = {}
    precision_per_class = {}

    for cls_id in range(14):
        tp = fp = fn = 0
        for pred, gt in zip(preds, targets):
            gt_mask = gt["labels"] == cls_id
            pr_mask = pred["labels"] == cls_id
            gt_boxes = gt["boxes"][gt_mask]
            pr_boxes = pred["boxes"][pr_mask]
            pr_scores = pred["scores"][pr_mask]

            if len(gt_boxes) == 0:
                fp += len(pr_boxes)
                continue
            if len(pr_boxes) == 0:
                fn += len(gt_boxes)
                continue

            order = np.argsort(-pr_scores)
            pr_boxes = pr_boxes[order]

            x1 = np.maximum(pr_boxes[:,None,0], gt_boxes[None,:,0])
            y1 = np.maximum(pr_boxes[:,None,1], gt_boxes[None,:,1])
            x2 = np.minimum(pr_boxes[:,None,2], gt_boxes[None,:,2])
            y2 = np.minimum(pr_boxes[:,None,3], gt_boxes[None,:,3])
            inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
            area_p = (pr_boxes[:,2]-pr_boxes[:,0]) * (pr_boxes[:,3]-pr_boxes[:,1])
            area_g = (gt_boxes[:,2]-gt_boxes[:,0]) * (gt_boxes[:,3]-gt_boxes[:,1])
            union = area_p[:,None] + area_g[None,:] - inter
            iou = np.where(union > 0, inter / union, 0.0)

            matched_gt = set()
            for pi in range(len(pr_boxes)):
                gi = int(np.argmax(iou[pi]))
                if iou[pi, gi] >= iou_thr and gi not in matched_gt:
                    tp += 1
                    matched_gt.add(gi)
                else:
                    fp += 1
            fn += len(gt_boxes) - len(matched_gt)

        recall_per_class[cls_id] = tp / (tp + fn + 1e-9) if (tp + fn) > 0 else float("nan")
        precision_per_class[cls_id] = tp / (tp + fp + 1e-9) if (tp + fp) > 0 else float("nan")

        total_tp += tp
        total_fp += fp
        total_fn += fn

    weights_cls = []
    w_recalls, w_precisions = [], []

    for cls_id in range(14):
        n_gt = sum(
            int((gt["labels"] == cls_id).sum())
            for gt in targets
        )
        r_cls = recall_per_class.get(cls_id, float("nan"))
        p_cls = precision_per_class.get(cls_id, float("nan"))

        if np.isnan(r_cls) or np.isnan(p_cls) or n_gt == 0:
            continue

        weights_cls.append(n_gt)
        w_recalls.append(r_cls)
        w_precisions.append(p_cls)

    weights_arr = np.array(weights_cls, dtype=float)
    weights_arr /= weights_arr.sum()

    recall = float(np.dot(weights_arr, w_recalls))
    precision = float(np.dot(weights_arr, w_precisions))
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    f2 = 5 * precision * recall / (4 * precision + recall + 1e-9)

    return {
        "recall": recall,
        "precision": precision,
        "f1": f1,
        "f2": f2,
        "map": map_val,
        "map_per_class": {i: float(v) for i, v in enumerate(map_pc)},
        "recall_per_class": recall_per_class,
        "precision_per_class": precision_per_class,
    }


def print_metrics(m: Dict, title: str = "", iou_thr: float = 0.4) -> None:
    print(f"\n{'='*60}")
    if title:
        print(f"  {title}")
        print(f"  {'-'*56}")
    print(f"  Recall:       {m['recall']:.4f}   ← главная метрика")
    print(f"  Precision:    {m['precision']:.4f}")
    print(f"  F1:           {m['f1']:.4f}")
    print(f"  F2:           {m['f2']:.4f}   ← акцент на Recall (β=2)")
    print(f"  mAP@{iou_thr}:    {m['map']:.4f}")
    print(f"  {'-'*56}")
    print(f"  {'Класс':<28} {'AP':>6}  {'Recall':>7}")
    print(f"  {'-'*56}")
    for cls_id in range(14):
        ap = m["map_per_class"].get(cls_id, float("nan"))
        rec = m["recall_per_class"].get(cls_id, float("nan"))
        ap_str  = f"{ap:.4f}"  if not np.isnan(ap)  else "   n/a"
        rec_str = f"{rec:.4f}" if not np.isnan(rec) else "   n/a"
        print(f"  {CLASSES.get(cls_id,'?'):<28} {ap_str:>6}  {rec_str:>7}")
    print(f"{'='*60}")


# Инференс батчами

In [ ]:
def run_inference(ensemble: Ensemble, img_dir: Path, csv_path: Path,
                  batch_size: int = 2, max_samples: int = None) -> tuple:
    """
    Читает CSV, запускает инференс батчами.
    Возвращает (preds, targets).
    """
    df = pd.read_csv(csv_path)

    items = []
    for iid, grp in df.groupby("image_id"):
        img_path = next(
            (img_dir / f"{iid}{ext}"
             for ext in [".png", ".jpg", ".jpeg", ".dicom", ".dcm"]
             if (img_dir / f"{iid}{ext}").exists()), None
        )
        if img_path is None:
            continue

        boxes, labels = [], []
        for _, row in grp.iterrows():
            cls = row.get("class_name", "No finding")
            if cls == "No finding":
                continue
            boxes.append([row.x_min, row.y_min, row.x_max, row.y_max])
            labels.append(CLASS_MAP.get(cls, 0))

        items.append({
            "image_id": str(iid),
            "img_path": img_path,
            "boxes":    np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4)),
            "labels":   np.array(labels, dtype=int),
        })
        if max_samples and len(items) >= max_samples:
            break

    all_preds, all_targets = [], []
    for i in tqdm(range(0, len(items), batch_size), desc=f"Инференс [{csv_path.name}]"):
        batch = items[i : i + batch_size]
        b_det = [load_image_for_detectors(b["img_path"]) for b in batch]
        b_ids = [b["image_id"] for b in batch]

        all_preds.extend(ensemble.predict(b_det, b_ids))
        all_targets.extend([{"image_id": b["image_id"],
                              "boxes":    b["boxes"],
                              "labels":   b["labels"]} for b in batch])
        del b_det

    return all_preds, all_targets


# Инициализация ансамбля

In [ ]:
ensemble = Ensemble(CFG)

# Оценка на val

In [ ]:
train_preds, train_targets = run_inference(ensemble, IMG_DIR, VAL_CSV, BATCH_SIZE, MAX_SAMPLES)
train_metrics = compute_metrics(train_preds, train_targets, IOU_THR)
print_metrics(train_metrics, title="TRAIN", iou_thr=IOU_THR)

# Оценка ансамбля и отдельных детекторов

Единый протокол метрик в стиле Ultralytics

In [ ]:
# conf=0.001, NMS iou=0.6, AP интерполяция по 101 точке recall, IoU@0.5

PROTO_CONF = 0.001 # минимальный порог
PROTO_NMS_IOU = 0.60 # NMS перед метриками
PROTO_AP_IOU = 0.50 # IoU для матчинга TP/FP
PROTO_RECALL_POINTS = np.linspace(0, 1, 101)  # 101-точечная интерполяция


def apply_nms(boxes: np.ndarray, scores: np.ndarray,
              labels: np.ndarray, iou_thr: float):
    if len(boxes) == 0:
        return boxes, scores, labels
    keep_all = []
    for cls_id in np.unique(labels):
        mask = labels == cls_id
        idx = nms(
            torch.tensor(boxes[mask], dtype=torch.float32),
            torch.tensor(scores[mask], dtype=torch.float32),
            iou_thr,
        ).numpy()
        keep_all.extend(np.where(mask)[0][idx].tolist())
    keep_all = sorted(keep_all, key=lambda i: scores[i], reverse=True)
    return boxes[keep_all], scores[keep_all], labels[keep_all]


def compute_ap_101(recalls: np.ndarray, precisions: np.ndarray) -> float:
    ap = 0.0
    for thr in PROTO_RECALL_POINTS:
        prec_at_thr = precisions[recalls >= thr]
        ap += prec_at_thr.max() if len(prec_at_thr) else 0.0
    return ap / len(PROTO_RECALL_POINTS)


def compute_metrics_proto(preds: list, targets: list,
                          ap_iou: float = PROTO_AP_IOU) -> dict:
    """
    Единый протокол метрик.
    preds  : [{image_id, boxes (N,4), scores (N,), labels (N,)}]
    targets: [{image_id, boxes (M,4), labels (M,)}]
    Возвращает: precision, recall, f1, f2, map50, ap_per_class, recall_per_class
    """
    n_classes = 14
    cls_tp = {c: [] for c in range(n_classes)}
    cls_fp = {c: [] for c in range(n_classes)}
    cls_n_gt = {c: 0  for c in range(n_classes)}

    gt_by_id = {t["image_id"]: t for t in targets}

    for pred in preds:
        iid = pred["image_id"]
        boxes = pred["boxes"]
        scores = pred["scores"]
        labels = pred["labels"]
        gt = gt_by_id.get(iid, {"boxes": np.zeros((0,4)), "labels": np.zeros(0,dtype=int)})

        for cls_id in range(n_classes):
            gt_mask = gt["labels"] == cls_id
            pr_mask = labels == cls_id
            gt_boxes = gt["boxes"][gt_mask]
            pr_boxes = boxes[pr_mask]
            pr_scores= scores[pr_mask]

            cls_n_gt[cls_id] += len(gt_boxes)

            if len(pr_boxes) == 0:
                continue

            order = np.argsort(-pr_scores)
            pr_boxes = pr_boxes[order]
            pr_scores = pr_scores[order]

            matched_gt = set()
            for pb, ps in zip(pr_boxes, pr_scores):
                if len(gt_boxes) == 0:
                    cls_fp[cls_id].append(ps)
                    continue

                x1 = np.maximum(pb[0], gt_boxes[:, 0])
                y1 = np.maximum(pb[1], gt_boxes[:, 1])
                x2 = np.minimum(pb[2], gt_boxes[:, 2])
                y2 = np.minimum(pb[3], gt_boxes[:, 3])
                inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
                area_p = (pb[2]-pb[0]) * (pb[3]-pb[1])
                area_g = (gt_boxes[:,2]-gt_boxes[:,0]) * (gt_boxes[:,3]-gt_boxes[:,1])
                union = area_p + area_g - inter
                iou = np.where(union > 0, inter / union, 0.0)

                best_gi = int(np.argmax(iou))
                if iou[best_gi] >= ap_iou and best_gi not in matched_gt:
                    cls_tp[cls_id].append(ps)
                    matched_gt.add(best_gi)
                else:
                    cls_fp[cls_id].append(ps)

    ap_per_class = {}
    recall_per_class = {}
    class_precision = {}
    class_recall = {}

    for cls_id in range(n_classes):
        n_gt = cls_n_gt[cls_id]
        tps = np.array(cls_tp[cls_id])
        fps = np.array(cls_fp[cls_id])

        if n_gt == 0 and len(tps) == 0:
            ap_per_class[cls_id] = float("nan")
            recall_per_class[cls_id] = float("nan")
            class_precision[cls_id] = float("nan")
            class_recall[cls_id] = float("nan")
            continue

        all_scores = np.concatenate([tps, fps]) if len(fps) else tps
        is_tp = np.array([1]*len(tps) + [0]*len(fps))
        order = np.argsort(-all_scores)
        is_tp = is_tp[order]

        cum_tp = np.cumsum(is_tp)
        cum_fp = np.cumsum(1 - is_tp)

        recalls = cum_tp / (n_gt + 1e-9)
        precisions = cum_tp / (cum_tp + cum_fp + 1e-9)

        ap_per_class[cls_id] = compute_ap_101(recalls, precisions)
        recall_per_class[cls_id] = float(cum_tp[-1]) / (n_gt + 1e-9) if len(cum_tp) else 0.0
        f1_curve = 2 * precisions * recalls / (precisions + recalls + 1e-9)
        best_idx = int(np.argmax(f1_curve))
        class_precision[cls_id] = float(precisions[best_idx])
        class_recall[cls_id] = float(recalls[best_idx])

    valid_ap  = [v for v in ap_per_class.values() if not np.isnan(v)]
    valid_p   = [v for v in class_precision.values() if not np.isnan(v)]
    valid_r   = [v for v in class_recall.values() if not np.isnan(v)]

    mp  = float(np.mean(valid_p)) if valid_p else 0.0
    mr  = float(np.mean(valid_r)) if valid_r else 0.0
    map50 = float(np.mean(valid_ap)) if valid_ap else 0.0
    f1  = 2 * mp * mr / (mp + mr + 1e-9)
    f2  = 5 * mp * mr / (4 * mp + mr + 1e-9)

    return {
        "precision": mp,
        "recall": mr,
        "f1": f1,
        "f2": f2,
        "map50": map50,
        "ap_per_class": ap_per_class,
        "recall_per_class": recall_per_class,
    }


In [ ]:
df_proto = pd.read_csv(VAL_CSV)
eval_items, eval_targets = [], []

for iid, grp in df_proto.groupby("image_id"):
    img_path = next(
        (IMG_DIR / f"{iid}{ext}" for ext in [".png", ".jpg", ".jpeg"]
         if (IMG_DIR / f"{iid}{ext}").exists()), None
    )
    if img_path is None:
        continue

    boxes, labels = [], []
    for _, row in grp.iterrows():
        cls = row.get("class_name", "No finding")
        if cls == "No finding":
            continue
        boxes.append([row.x_min, row.y_min, row.x_max, row.y_max])
        labels.append(CLASS_MAP.get(cls, 0))

    item = {
        "image_id": str(iid),
        "img_path": img_path,
        "boxes": np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4)),
        "labels": np.array(labels, dtype=int),
    }
    eval_items.append(item)
    eval_targets.append({
        "image_id": str(iid),
        "boxes": item["boxes"],
        "labels": item["labels"],
    })

print(f"Снимков: {len(eval_items)},  GT боксов: {sum(len(t['boxes']) for t in eval_targets)}")

In [ ]:
_saved = {k: ensemble.cfg[k] for k in ("yolov8_conf", "faster_rcnn_conf", "rt_detr_conf")}
ensemble.update(yolov8_conf=PROTO_CONF, faster_rcnn_conf=PROTO_CONF, rt_detr_conf=PROTO_CONF)

proto_preds = {}
for i in tqdm(range(0, len(eval_items), BATCH_SIZE), desc="Инференс (единый протокол)"):
    batch = eval_items[i : i + BATCH_SIZE]
    b_det = [load_image_for_detectors(b["img_path"]) for b in batch]
    b_ids = [b["image_id"] for b in batch]

    for name, model_obj in [("YOLOv8",     ensemble.yolo),
                             ("FasterRCNN", ensemble.faster_rcnn),
                             ("RT-DETR",    ensemble.rt_detr)]:
        for p in model_obj.predict(b_det, b_ids):
            boxes, scores, labels = apply_nms(
                p["boxes"], p["scores"], p["labels"], PROTO_NMS_IOU
            )
            proto_preds.setdefault(name, []).append({
                "image_id": p["image_id"],
                "img_w":    p["img_w"],
                "img_h":    p["img_h"],
                "boxes":    boxes,
                "scores":   scores,
                "labels":   labels,
            })
    del b_det

ensemble.update(**_saved)

# Ансамбль
fused_proto = run_wbf(
    preds_per_model=[proto_preds["YOLOv8"], proto_preds["FasterRCNN"], proto_preds["RT-DETR"]],
    weights=[CFG["w_yolov8"], CFG["w_faster_rcnn"], CFG["w_rt_detr"]],
    iou_thr=CFG["wbf_iou_thr"],
    skip_thr=CFG["wbf_skip_thr"],
    final_thr=CFG["final_score_thr"],
)
proto_preds["Ensemble (WBF)"] = [
    {"image_id": f["image_id"], "boxes": f["boxes"],
     "scores": f["scores"], "labels": f["labels"]}
    for f in fused_proto
]

# Метрики
results_proto = {}
for name, preds in proto_preds.items():
    results_proto[name] = compute_metrics_proto(preds, eval_targets)

# Построение сводной таблицы
print(f"\n{'─'*66}")
print(f"  {'Модель':<18} {'P':>7} {'R':>7} {'F1':>7} {'F2':>7} {'mAP@50':>8}")
print(f"{'─'*66}")
for name, m in results_proto.items():
    sep = f"\n  {'─'*62}" if name == "Ensemble (WBF)" else ""
    print(f"{sep}")
    print(f"  {name:<18} {m['precision']:>7.4f} {m['recall']:>7.4f} "
          f"{m['f1']:>7.4f} {m['f2']:>7.4f} {m['map50']:>8.4f}")
print(f"{'─'*66}")
print(f"  Протокол: conf≥{PROTO_CONF}  NMS iou={PROTO_NMS_IOU}  "
      f"AP@IoU={PROTO_AP_IOU}  101-point interpolation")

# bar chart
metrics_show = ["precision", "recall", "f1", "f2", "map50"]
labels_show  = ["Precision", "Recall", "F1", "F2", "mAP@50"]
colors_names = {"YOLOv8": "steelblue", "FasterRCNN": "tomato",
                "RT-DETR": "seagreen", "Ensemble (WBF)": "mediumpurple"}

n_models = len(results_proto)
x    = np.arange(len(metrics_show))
w    = 0.18
offs = np.linspace(-(n_models-1)/2*w, (n_models-1)/2*w, n_models)

fig, ax = plt.subplots(figsize=(12, 4))
for (name, m), offset in zip(results_proto.items(), offs):
    vals = [m[metric] for metric in metrics_show]
    bars = ax.bar(x + offset, vals, width=w, label=name,
                  color=colors_names[name], alpha=0.85, edgecolor="grey")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(labels_show)
ax.set_ylim(0, min(1.0, max(m[mt] for m in results_proto.values()
                             for mt in metrics_show) + 0.13))
ax.set_ylabel("Значение")
ax.set_title(f"Единый протокол метрик  "
             f"(conf≥{PROTO_CONF}, NMS iou={PROTO_NMS_IOU}, AP@{PROTO_AP_IOU})")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# mAP@50 по классам
fig, ax = plt.subplots(figsize=(13, 4))
x_cls    = np.arange(14)
w_cls    = 0.18
offs_cls = np.linspace(-(n_models-1)/2*w_cls, (n_models-1)/2*w_cls, n_models)

for (name, m), offset in zip(results_proto.items(), offs_cls):
    ap_vals = [m["ap_per_class"].get(i, float("nan")) for i in range(14)]
    ax.bar(x_cls + offset, ap_vals, width=w_cls, label=name,
           color=colors_names[name], alpha=0.8, edgecolor="none")

ax.set_xticks(x_cls)
ax.set_xticklabels([CLASSES.get(i, str(i)) for i in range(14)],
                   rotation=35, ha="right", fontsize=8)
ax.set_ylabel("AP@50")
ax.set_title("AP@50 по классам — единый протокол")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()